In [79]:
from proj_utils import *

import os
import pickle
import gc
from os.path import join
from copy import deepcopy
from collections import defaultdict, Counter
from itertools import combinations
from datetime import datetime
from dateutil.relativedelta import relativedelta
import ipywidgets as widgets
from IPython.display import display
from collections import OrderedDict

import numpy as np
import pandas as pd
from tqdm import tqdm

import sklearn as sk
from scipy.optimize import linear_sum_assignment
from scipy.optimize import curve_fit
from sklearn.metrics.pairwise import cosine_similarity

import torch
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import mpltern
import ipywidgets as widgets
from ipywidgets import interact, interact_manual

#import bertopic
from proj_utils import *

In [89]:
collection_name = 'Motherjones'
model_name = MODEL_NAMES[collection_name]

start_date, end_date = DATE_RANGES[collection_name]
num_topics = NUM_TOPICS[collection_name]

In [90]:
def aggregate_save(tmp_df, comment_date_cutoff):

    comment_createdAt_list = []
    comment_id_list = []
    comment_topics_list = []
    comment_embeddings_list = []
    comment_sentiments_list = []
    topic_freq = Counter({i: 0 for i in range(-1, num_topics)})
    
    for article in tmp_df.itertuples():
        filtered_index = np.where(np.array([(comment_createdAt - article.createdAt).days for comment_createdAt in article.comment_createdAt]) < comment_date_cutoff)[0]
        comment_createdAt_list.append([article.comment_createdAt[i] for i in filtered_index])
        comment_id_list.append([article.comment_id[i] for i in filtered_index])
        comment_topics = [article.comment_topics[i] for i in filtered_index]
        comment_topics_list.append(comment_topics)
        topic_freq.update(comment_topics)
        comment_embeddings_list.append([article.comment_embeddings[i] for i in filtered_index])
        comment_sentiments_list.append([article.comment_sentiments[i] for i in filtered_index])
        
    tmp_df = tmp_df.assign(comment_id=comment_id_list, comment_topics=comment_topics_list, comment_createdAt=comment_createdAt_list, comment_embeddings=comment_embeddings_list, comment_sentiments=comment_sentiments_list)

    topic_embedding_list = [[] for _ in range(num_topics+1)] # including -1
    topic_sentiment_list = [[] for _ in range(num_topics+1)] # including -1
    topic_mean_embedding_list = [[] for _ in range(num_topics+1)] # including -1
    topic_mean_sentiment_list = [[] for _ in range(num_topics+1)] # including -1

    for article in tmp_df.itertuples():
        for i in range(num_topics+1):  # including -1  
            comment_index = np.where(np.array(article.comment_topics) == i-1)[0]
            if len(comment_index)>0:
                topic_embedding_list[i].append(np.array(article.comment_embeddings)[comment_index])
                topic_sentiment_list[i].append(np.array(article.comment_sentiments)[comment_index])
                    
    for i in range(num_topics+1): 
        assert len(topic_embedding_list[i]) == len(topic_sentiment_list[i])
        
        if len(topic_embedding_list[i])>0:
            averaged_embedding = np.vstack(topic_embedding_list[i]).mean(axis=0)
        else:
            averaged_embedding = np.zeros(384)  # embedding shape
            
        if len(topic_sentiment_list[i])>0:
            averaged_sentiment = np.vstack(topic_sentiment_list[i]).mean(axis=0)
        else:
            averaged_sentiment = np.zeros(11)  # sentiment shape
        
        topic_mean_embedding_list[i] = averaged_embedding 
        topic_mean_sentiment_list[i] = averaged_sentiment
        
    # sort topic_freq by key and get values
    topic_freq = [topic_freq[key] for key in sorted(topic_freq.keys())]
    summary_df = pd.DataFrame({'topic_freq': topic_freq, 'topic_mean_embedding': topic_mean_embedding_list, 'topic_mean_sentiment': topic_mean_sentiment_list})

    summary_df_csv = pd.DataFrame({'topic_freq': topic_freq})
    tme = pd.DataFrame(topic_mean_embedding_list, columns=['e'+str(i) for i in range(384)])
    tms = pd.DataFrame(topic_mean_sentiment_list, columns=['s'+str(i) for i in range(11)])
    summary_df_csv = pd.concat([summary_df_csv, tme, tms], axis=1)
    
    return summary_df, summary_df_csv

In [91]:
aggregate_days_list = [3, 7, 30]
comment_threshold = 10
comment_date_cutoff_dict = {3: 1, 7: 2, 30: 7}

file_names = os.listdir(join('article', collection_name.lower(), model_name, 'articles_by_day'))
keys = sorted([file_name.split('.')[0] for file_name in file_names], key=lambda x: datetime.strptime(x, '%Y-%m-%d'))
start = datetime.strptime(keys[0], '%Y-%m-%d')

strat_end_dict = {aggregate_day: (start, start + relativedelta(days=aggregate_day)) for aggregate_day in aggregate_days_list}
topic_tuple_dict_dict = {}
article_topics = {}
title_topic_num_list = []
title_constant_list = []

tmp_list_df = {aggregate_day : [] for aggregate_day in aggregate_days_list}
len_list_dict = {aggregate_day: {} for aggregate_day in aggregate_days_list}

# make folders under /data/collmind/article
for aggregate_day in aggregate_days_list:
    os.makedirs(join('/data', 'collmind', 'article', collection_name.lower(), str(aggregate_day)+'d'), exist_ok=True)
    os.makedirs(join('/data', 'collmind', 'article', collection_name.lower(), str(aggregate_day)+'d', 'csv'), exist_ok=True)
    os.makedirs(join('/data', 'collmind', 'article', collection_name.lower(), str(aggregate_day)+'d', 'df'), exist_ok=True)

for key in keys:
    print(key)
    articles = pd.read_parquet(join('article', collection_name.lower(), model_name, 'articles_by_day', key +'.parquet'))
    articles = articles[articles['comment_id'].apply(len) > comment_threshold]
    
    for aggregate_day in aggregate_days_list:
        
        tmp_list_df[aggregate_day].append(articles)
        
        if datetime.strptime(key, '%Y-%m-%d') >= strat_end_dict[aggregate_day][1]:
            len_list_dict[aggregate_day][strat_end_dict[aggregate_day][0].strftime('%Y-%m-%d')] = len(tmp_list_df[aggregate_day])
            tmp_df = pd.concat(tmp_list_df[aggregate_day])
            
            if len(tmp_df) > 0:
                tmp_df = pd.concat(tmp_list_df[aggregate_day])
                
                topic_tuple_dict = defaultdict(list)
                for article in tmp_df.itertuples():
                    topic_tuple = tuple(sorted(article.topic_num[:3]))
                    topic_tuple_dict[topic_tuple].append((article._1, article.createdAt.strftime('%Y-%m-%d')))
                topic_tuple_dict_dict[strat_end_dict[aggregate_day][0].strftime('%Y-%m-%d')] = topic_tuple_dict
                
                title_constant_list.append(len(tmp_df))
                title_topic_num_list.append(np.zeros((num_topics, 3)))
                for t in range(num_topics):
                    for i in range(3):
                        title_topic_num_list[-1][t][i] = len(tmp_df[np.vstack(tmp_df['topic_num'])[:, i] == t])
                
                summary_df, summary_df_csv = aggregate_save(tmp_df, comment_date_cutoff_dict[aggregate_day])
                # save summary_df
                summary_df.to_parquet(join('/data', 'collmind', 'article', collection_name.lower(), str(aggregate_day)+'d', 'df', strat_end_dict[aggregate_day][0].strftime('%Y-%m-%d') + '.parquet'))
                summary_df_csv.to_csv(join('/data', 'collmind', 'article', collection_name.lower(), str(aggregate_day)+'d', 'csv', strat_end_dict[aggregate_day][0].strftime('%Y-%m-%d') + '.csv'))
                tmp_list_df[aggregate_day] = []
            
            strat_end_dict[aggregate_day] = (strat_end_dict[aggregate_day][1], strat_end_dict[aggregate_day][1] + relativedelta(days=aggregate_day))
          
for aggregate_day in aggregate_days_list:
        
    len_list_dict[aggregate_day][strat_end_dict[aggregate_day][0].strftime('%Y-%m-%d')] = len(tmp_list_df[aggregate_day])
    tmp_df = pd.concat(tmp_list_df[aggregate_day])
    
    if len(tmp_df) > 0:
        tmp_df = pd.concat(tmp_list_df[aggregate_day])
        
        topic_tuple_dict = defaultdict(list)
        for article in tmp_df.itertuples():
            topic_tuple = tuple(sorted(article.topic_num[:3]))
            topic_tuple_dict[topic_tuple].append((article._1, article.createdAt.strftime('%Y-%m-%d')))
        topic_tuple_dict_dict[strat_end_dict[aggregate_day][0].strftime('%Y-%m-%d')] = topic_tuple_dict
        
        title_constant_list.append(len(tmp_df))
        title_topic_num_list.append(np.zeros((num_topics, 3)))
        for t in range(num_topics):
            for i in range(3):
                title_topic_num_list[-1][t][i] = len(tmp_df[np.vstack(tmp_df['topic_num'])[:, i] == t])
        
        summary_df, summary_df_csv = aggregate_save(tmp_df, comment_date_cutoff_dict[aggregate_day])
        # save summary_df
        summary_df.to_parquet(join('/data', 'collmind', 'article', collection_name.lower(), str(aggregate_day)+'d', 'df', strat_end_dict[aggregate_day][0].strftime('%Y-%m-%d') + '.parquet'))
        summary_df_csv.to_csv(join('/data', 'collmind', 'article', collection_name.lower(), str(aggregate_day)+'d', 'csv', strat_end_dict[aggregate_day][0].strftime('%Y-%m-%d') + '.csv'))
        tmp_list_df[aggregate_day] = []
    
            
result_dict = {'topic_tuple_dict_dict': topic_tuple_dict_dict, 'len_list_dict': len_list_dict, 'title_constant_list': title_constant_list, 'title_topic_num_list': title_topic_num_list}

# save num_elements
with open(join('/data', 'collmind', 'article', collection_name.lower(), 'result_dict.pkl'), 'wb') as f:
    pickle.dump(result_dict, f)

2012-06-01
2012-06-02
2012-06-03
2012-06-04
2012-06-05
2012-06-06
2012-06-07
2012-06-08
2012-06-09
2012-06-10
2012-06-11
2012-06-12
2012-06-13
2012-06-14
2012-06-15
2012-06-16
2012-06-17
2012-06-18
2012-06-19
2012-06-20
2012-06-21
2012-06-22
2012-06-23
2012-06-24
2012-06-25
2012-06-26
2012-06-27
2012-06-28
2012-06-29
2012-06-30
2012-07-01
2012-07-02
2012-07-03
2012-07-04
2012-07-05
2012-07-06
2012-07-07
2012-07-08
2012-07-09
2012-07-10
2012-07-11
2012-07-12
2012-07-13
2012-07-14
2012-07-15
2012-07-16
2012-07-17
2012-07-18
2012-07-19
2012-07-20
2012-07-21
2012-07-22
2012-07-23
2012-07-24
2012-07-25
2012-07-26
2012-07-27
2012-07-28
2012-07-29
2012-07-30
2012-07-31
2012-08-01
2012-08-02
2012-08-03
2012-08-04
2012-08-05
2012-08-06
2012-08-07
2012-08-08
2012-08-09
2012-08-10
2012-08-11
2012-08-12
2012-08-13
2012-08-14
2012-08-15
2012-08-16
2012-08-17
2012-08-18
2012-08-19
2012-08-20
2012-08-21
2012-08-22
2012-08-23
2012-08-24
2012-08-25
2012-08-26
2012-08-27
2012-08-28
2012-08-29
2012-08-30

In [78]:
result_dict = {'topic_tuple_dict_dict': topic_tuple_dict_dict, 'len_list_dict': len_list_dict, 'title_constant_list': title_constant_list, 'title_topic_num_list': title_topic_num_list}

# save num_elements
with open(join('/data', 'collmind', 'article', collection_name.lower(), 'result_dict.pkl'), 'wb') as f:
    pickle.dump(result_dict, f)